# 02 · Data Preparation  (CRISP-DM phase 3)

Steps: harmonise names/units → aggregate markets to **province-month** → build a complete monthly grid → **seasonal imputation of the remaining gaps, with an explicit `is_imputed` flag** → feature engineering → save.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
import prep
FIG = os.path.join(ROOT, "reports", "figures"); os.makedirs(FIG, exist_ok=True)
RAW = os.path.join(ROOT, "data", "wfp_food_prices_zmb.csv")
PROC = os.path.join(ROOT, "data", "processed"); os.makedirs(PROC, exist_ok=True)
MODELS = os.path.join(ROOT, "models"); os.makedirs(MODELS, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "figure.autolayout": True})
def save(name):
    plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.show()


## 3.1 Harmonise commodity names and units

In [ ]:
raw = pd.read_csv(RAW)
h = prep.harmonise(raw)
display(h.groupby(["commodity_std","unit_std"]).size().rename("records").reset_index())

## 3.2 Aggregate to province level
Median across all markets in the province for each month. `n_markets` is kept as a feature (reliability of the aggregate).

In [ ]:
agg = prep.aggregate_province(h)
print(agg.shape); display(agg.head())

## 3.3 Monthly grid + seasonal imputation (flagged)

* The grid runs from the first to the last *real* observation of each series (we never extrapolate beyond the last real record – so the "last observation date" is always a real one).
* Interior gaps are filled with **seasonal imputation**: linear interpolation of the log-price trend + a (shrunk) month-of-year seasonal effect estimated from that series' own history.
* **Every imputed value gets `is_imputed = 1`**, and that flag is passed to the models as features (`is_imputed_t`, `imp_share_6`) and used to exclude imputed targets from evaluation (we never score a model against a number we invented).
* Gaps longer than 6 months (e.g. between survey eras) are *not* bridged: the series is split into segments.
* Series with < 12 real observations are dropped.

In [ ]:
panel = prep.build_monthly_panel(agg)
print("rows:", len(panel), "| series (incl. old segments):", panel.series_id.nunique(),
      "| latest-segment series:", panel.loc[panel.is_latest_segment==1,"series_id"].nunique())
print("imputed share: %.2f%%" % (panel.is_imputed.mean()*100))
display(panel[panel.is_imputed==1].head())

In [ ]:
# Example of imputation: choose the series with most imputed months
sid = panel.groupby("series_id").is_imputed.sum().idxmax()
g = panel[panel.series_id==sid]
fig, ax = plt.subplots(figsize=(10,3.6))
ax.plot(g.month, g.price, color="#2a6f97", lw=1.2, label="series used by model")
ax.scatter(g.month[g.is_imputed==0], g.price_raw[g.is_imputed==0], s=12, color="#2a6f97", label="observed")
ax.scatter(g.month[g.is_imputed==1], g.price[g.is_imputed==1], s=45, color="#e63946", marker="x", label="imputed (flag = 1)")
ax.set_title(f"Seasonal imputation example – {sid}"); ax.legend()
save("02_imputation_example.png")

In [ ]:
imp = panel.groupby("commodity").is_imputed.mean().sort_values()*100
fig, ax = plt.subplots(figsize=(7,5)); imp.plot.barh(ax=ax, color="#e07a5f")
ax.set_title("Share of months imputed, by commodity (%)")
save("02_imputed_share_by_commodity.png")

## 3.4 Feature engineering (all computed only from information available at month *t*)

| Group | Features | Rationale |
|---|---|---|
| `lags` | log-price k months ago relative to now, k ∈ {1,2,3,6,12} | autoregression, annual cycle |
| `momentum` | 1/3/6-month log change | trend |
| `rolling` | mean/std of monthly changes (3, 6 months) | local drift and volatility |
| `seasonal` | sin/cos of month, **seasonal index** = past average change into the target calendar month | seasonality |
| `imputation` | `is_imputed_t`, share of imputed months in last 6, `n_markets_t` | **lets the algorithm know which observations are imputed** and how reliable the aggregate is |
| `series_id` | province code, commodity code, log price level | series identity |
| `cross_section` | mean momentum of the same commodity in the *other* provinces, price relative to other provinces | prices co-move nationally |

**Target:** `y = log(price_{t+1}) − log(price_t)`; predicted price = `price_t · exp(ŷ)`.

In [ ]:
pc, cc = prep.codes(panel)
F = prep.make_features(panel, pc, cc)
print(F.shape); display(F[["series_id","month","price","is_imputed_t","y","y_price","y_imputed"]+prep.ALL_FEATURES[:6]].head())
print("missing share per feature:"); print(F[prep.ALL_FEATURES].isna().mean().round(3).to_string())

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,3.3))
F.y.clip(-.6,.6).plot.hist(bins=80, ax=ax[0], color="#2a6f97"); ax[0].set_title("Target: next-month log price change (clipped for display)")
F.groupby(F.month.dt.month).y.mean().mul(100).plot.bar(ax=ax[1], color="#8fb339"); ax[1].set_title("Mean next-month change by current calendar month (%)")
save("02_target_distribution_and_seasonality.png")

In [ ]:
panel.to_csv(os.path.join(PROC, "panel_monthly.csv"), index=False)
F.to_csv(os.path.join(PROC, "features.csv"), index=False)
json.dump({"province_codes": pc, "commodity_codes": cc}, open(os.path.join(PROC, "codes.json"), "w"), indent=1)
print("saved panel_monthly.csv, features.csv, codes.json")

Proceed to notebook 03 (modelling and ablation).